# ComfyUI on Kaggle

This puts ComfyUI on a free Kaggle GPU. You generate images in the browser. Nothing installs on your laptop.

There are **no forms, ticks, or hidden menus**. You edit **one cell**, then press **Run** on the rest.

---

## Before any Run

Right side of the page → **Session options**:

1. Accelerator = **GPU T4**  
   (use **GPU T4 x2** only if you want two GPUs)
2. Internet = **On**
3. Save, wait until it says connected

---

## How to use this notebook

1. Open **Your choices** (next cell). Change only what you need.
2. Run that cell.
3. Run the next cells **in order**: Check → Install → Get models → Start.
4. When Start prints a `https://….trycloudflare.com` line: **copy it, new tab, paste, Enter**.
5. Do **not** click the link inside Kaggle (that shows “Access denied”).
6. Leave the Start cell running.

| I want… | In Your choices set |
|---|---|
| First try, download a small model | `HOW = "download"`  (DreamShaper is already on) |
| I already clicked **Add Data** | `HOW = "dataset"` |
| I have a name like `owner/dataset` | `HOW = "kagglehub"` and fill `KAGGLE_DATASET_SLUG` |
| Two GPUs, VAE+text on 0, UNet on 1 | Accelerator **T4 x2**, then `USE_MULTIGPU = True` |

If a cell turns red, read the line that starts with **FIX**.



## 1 · Your choices

This is the **only** cell you should edit.

- Change `HOW` to match your files.
- Leave the rest unless you know you need it.
- Then press Run **once**.



In [ ]:
# 1 · Your choices  —  edit this, then Run once
# ------------------------------------------------------------
# HOW to get models  (pick one word, keep the quotes)
#   "download"   first time — fetch DreamShaper from the internet
#   "dataset"    you already clicked Add Data on the right
#   "kagglehub"  you will paste owner/dataset-name below
# ------------------------------------------------------------
HOW = "download"

# If HOW = "dataset": which attached folder?  "auto" = all of them
DATASET_NAME = "auto"

# If HOW = "kagglehub": paste a public Kaggle name
KAGGLE_DATASET_SLUG = ""      # example: owner/my-comfyui-models
KAGGLE_MODEL_HANDLE = ""      # optional Kaggle Model handle

# If HOW = "download": True = get this file. Start with DreamShaper only.
GET_DREAMSHAPER = True        # ~2 GB   good first try
GET_SD15 = False              # ~4 GB
GET_SDXL = False              # ~7 GB
GET_SDXL_VAE = False
GET_LIGHTNING_LORA = False
GET_CONTROLNET_CANNY = False
GET_UPSCALER = True           # tiny, useful
GET_FLUX = False              # ~17 GB  skip on a first run
CUSTOM_URL = ""               # optional extra direct file link
CUSTOM_FOLDER = "checkpoints" # checkpoints / loras / vae / …

# Tokens — only if a download asks you to log in
HF_TOKEN = ""
CIVITAI_TOKEN = ""

# Two GPUs (Session options must be GPU T4 x2)
USE_MULTIGPU = False
VAE_ON = "cuda:0"
TEXT_ENCODER_ON = "cuda:0"
UNET_ON = "cuda:1"

# ------------------------------------------------------------
# You can stop editing here. Press Run.
# ------------------------------------------------------------
import json, os

CONFIG = "/kaggle/working/comfy_kaggle_config.json"
os.makedirs("/kaggle/working", exist_ok=True)

how = str(HOW).strip().lower()
if how not in ("download", "dataset", "kagglehub"):
    raise SystemExit('FIX  HOW must be "download" or "dataset" or "kagglehub"')

cfg = {}
if os.path.exists(CONFIG):
    with open(CONFIG) as f:
        cfg = json.load(f)

cfg.update({
    "how": how,
    "dataset_name": DATASET_NAME.strip() or "auto",
    "kaggle_dataset_slug": KAGGLE_DATASET_SLUG.strip().strip("/"),
    "kaggle_model_handle": KAGGLE_MODEL_HANDLE.strip().strip("/"),
    "get": {
        "dreamshaper": bool(GET_DREAMSHAPER),
        "sd15": bool(GET_SD15),
        "sdxl": bool(GET_SDXL),
        "sdxl_vae": bool(GET_SDXL_VAE),
        "lightning": bool(GET_LIGHTNING_LORA),
        "canny": bool(GET_CONTROLNET_CANNY),
        "upscaler": bool(GET_UPSCALER),
        "flux": bool(GET_FLUX),
    },
    "custom_url": CUSTOM_URL.strip(),
    "custom_folder": CUSTOM_FOLDER.strip() or "checkpoints",
    "hf_token": HF_TOKEN.strip(),
    "civitai_token": CIVITAI_TOKEN.strip(),
    "use_multigpu": bool(USE_MULTIGPU),
    "vae_on": VAE_ON.strip() or "cuda:0",
    "te_on": TEXT_ENCODER_ON.strip() or "cuda:0",
    "unet_on": UNET_ON.strip() or "cuda:1",
    "store_root": "/kaggle/working/ComfyUI_store",
    "comfy_path": "/kaggle/working/ComfyUI",
})
with open(CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)

print("-" * 56)
print("Choices saved. You do not need to edit any later cell.")
print("-" * 56)
print("HOW:         ", how)
if how == "download":
    on = [k for k, v in cfg["get"].items() if v]
    print("Will download:", ", ".join(on) or "(nothing — turn one GET_ line to True)")
    if cfg["custom_url"]:
        print("Plus URL:    ", cfg["custom_url"])
elif how == "dataset":
    print("Will use Add Data folders, name =", cfg["dataset_name"])
else:
    print("Dataset slug:", cfg["kaggle_dataset_slug"] or "(empty — fill KAGGLE_DATASET_SLUG)")
    print("Model handle:", cfg["kaggle_model_handle"] or "(optional)")
print("Multi-GPU:   ", "yes" if cfg["use_multigpu"] else "no")
print()
print("NEXT  Run the next cell: Check the machine")



## 2 · Check the machine

Press Run. You want GPU and Internet both OK.

If GPU fails: Session options → Accelerator → GPU T4 → Save → wait → run this cell again.



In [ ]:
# 2 · Check the machine  —  just press Run
import os, shutil, urllib.request

print("-" * 56)
print("Checking GPU, internet, disk")
print("-" * 56)
gpu_ok = net_ok = True

print("\nGPU")
try:
    import torch
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        for i in range(n):
            p = torch.cuda.get_device_properties(i)
            gb = p.total_memory / (1024 ** 3)
            print(f"  OK   GPU {i}: {torch.cuda.get_device_name(i)}  ({gb:.1f} GB)")
            if i == 0:
                os.environ["COMFY_VRAM_GB"] = f"{gb:.1f}"
        if n >= 2:
            print(f"  OK   {n} GPUs — you can set USE_MULTIGPU = True in Your choices")
    else:
        gpu_ok = False
        print("  FIX  No GPU. Session options → Accelerator → GPU T4 → Save")
except Exception as e:
    gpu_ok = False
    print("  FIX  Could not read GPU:", e)

print("\nInternet")
try:
    urllib.request.urlopen("https://github.com", timeout=12)
    print("  OK   Internet is on")
except Exception:
    net_ok = False
    print("  FIX  Session options → Internet → On → Save")

print("\nDisk")
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"  OK   {free/1e9:.0f} GB free")
if free < 20e9:
    print("       Low space. Prefer a Dataset. Do not turn on GET_FLUX.")

print("\nAdd Data folders")
inp = "/kaggle/input"
if os.path.isdir(inp) and os.listdir(inp):
    for name in sorted(os.listdir(inp)):
        print(f"  OK   {name}")
else:
    print("       none attached (fine if HOW is download or kagglehub)")

print()
print("Summary: GPU", "OK" if gpu_ok else "FIX", " · Internet", "OK" if net_ok else "FIX")
print("NEXT  Run the next cell: Install ComfyUI")



## 3 · Install ComfyUI

Press Run and wait **2–5 minutes**. This installs the app, not the art models.

You should see OK for: ComfyUI, packages, Manager, downloader, public link.



In [ ]:
# 3 · Install ComfyUI  —  just press Run
import json, os, shutil, subprocess, sys
from pathlib import Path

CONFIG = "/kaggle/working/comfy_kaggle_config.json"
COMFY = "/kaggle/working/ComfyUI"
STORE = "/kaggle/working/ComfyUI_store"
REPO = "https://github.com/comfyanonymous/ComfyUI.git"
MANAGER = "https://github.com/Comfy-Org/ComfyUI-Manager.git"
MG_REPO = "https://github.com/pollockjj/ComfyUI-MultiGPU.git"
SUBS = [
    "checkpoints", "loras", "vae", "controlnet", "clip", "clip_vision",
    "text_encoders", "unet", "diffusion_models", "upscale_models",
    "embeddings", "hypernetworks", "photomaker", "style_models",
    "diffusers", "gligen", "vae_approx", "ipadapter", "animatediff_models",
    "animatediff_motion_lora", "configs", "model_patches", "audio_encoders",
]

def load():
    if os.path.exists(CONFIG):
        with open(CONFIG) as f:
            return json.load(f)
    return {}

def save(**kw):
    cfg = load()
    cfg.update(kw)
    with open(CONFIG, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg

def run(cmd):
    subprocess.check_call(cmd)

cfg = load()
if not cfg:
    print("FIX  Run cell 1 (Your choices) first.")
    raise SystemExit("Run Your choices first")

print("-" * 56)
print("Installing ComfyUI")
print("-" * 56)

root = Path(STORE)
root.mkdir(parents=True, exist_ok=True)
for s in SUBS:
    (root / "models" / s).mkdir(parents=True, exist_ok=True)
for extra in ("output", "input", "custom_nodes", "user"):
    (root / extra).mkdir(parents=True, exist_ok=True)
save(store_root=str(root), comfy_path=COMFY)
print("OK   Folders ready at", root)

print("\nComfyUI")
if not os.path.isdir(COMFY):
    print("WAIT Downloading the app…")
    run(["git", "clone", "--depth", "1", REPO, COMFY])
else:
    print("OK   Already present — updating")
    subprocess.call(["git", "-C", COMFY, "pull", "--ff-only"])
print("OK   App is on disk")

os.chdir(COMFY)
print("\nPython packages (2–4 minutes the first time)")
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("OK   Packages ready")

print("\nManager (add nodes later from the UI)")
mgr = os.path.join(COMFY, "custom_nodes", "ComfyUI-Manager")
if not os.path.isdir(mgr):
    run(["git", "clone", "--depth", "1", MANAGER, mgr])
else:
    subprocess.call(["git", "-C", mgr, "pull", "--ff-only"])
req = os.path.join(mgr, "requirements.txt")
if os.path.isfile(req):
    run([sys.executable, "-m", "pip", "install", "-q", "-r", req])
print("OK   Manager ready")

if cfg.get("use_multigpu"):
    print("\nMulti-GPU node pack")
    dest = Path(COMFY) / "custom_nodes" / "ComfyUI-MultiGPU"
    if dest.exists():
        subprocess.call(["git", "-C", str(dest), "pull", "--ff-only"])
    else:
        run(["git", "clone", "--depth", "1", MG_REPO, str(dest)])
    req = dest / "requirements.txt"
    if req.is_file():
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("OK   ComfyUI-MultiGPU installed")
    print("     After launch: VAELoaderMultiGPU + CLIPLoaderMultiGPU on", cfg.get("vae_on"),
          "/ UNETLoaderMultiGPU on", cfg.get("unet_on"))
    print("     Do not use CheckpointLoaderSimple")

print("\nHelpers")
subprocess.call(["apt-get", "-y", "-qq", "update"], stdout=subprocess.DEVNULL)
subprocess.call(["apt-get", "-y", "-qq", "install", "aria2"], stdout=subprocess.DEVNULL)
print("OK   Fast downloader (aria2)")
if not shutil.which("cloudflared"):
    deb = "/tmp/cloudflared.deb"
    run(["wget", "-q", "-O", deb,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.call(["dpkg", "-i", deb])
print("OK   Public-link tool (cloudflared)")

print("\nOK   Install finished")
print("NEXT Run the next cell: Get models")



## 4 · Get models

Press Run. This cell reads **HOW** from Your choices:

- `download` — fetches the files you turned on (DreamShaper by default)
- `dataset` — uses folders from **Add Data** (no download)
- `kagglehub` — pulls the slug you pasted

Then it tells ComfyUI where those files live. You do not run a second linking cell.



In [ ]:
# 4 · Get models  —  just press Run
import json, os, shutil, subprocess, sys
from pathlib import Path
from urllib.parse import unquote, urlparse

CONFIG = "/kaggle/working/comfy_kaggle_config.json"
WEIGHT_EXT = {".safetensors", ".ckpt", ".pt", ".pth", ".bin", ".gguf", ".sft"}
SUBS = [
    "checkpoints", "loras", "vae", "controlnet", "clip", "clip_vision",
    "text_encoders", "unet", "diffusion_models", "upscale_models",
    "embeddings", "hypernetworks", "photomaker", "style_models",
    "diffusers", "gligen", "vae_approx", "ipadapter", "animatediff_models",
    "animatediff_motion_lora", "configs", "model_patches", "audio_encoders",
]

def load():
    with open(CONFIG) as f:
        return json.load(f)

def save(**kw):
    cfg = load()
    cfg.update(kw)
    with open(CONFIG, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        return os.environ.get(name, "") or ""

if not os.path.exists(CONFIG):
    raise SystemExit("FIX  Run cell 1 (Your choices) first")

cfg = load()
how = cfg.get("how") or "download"
comfy = Path(cfg.get("comfy_path") or "/kaggle/working/ComfyUI")
store = Path(cfg.get("store_root") or "/kaggle/working/ComfyUI_store")
models_root = store / "models"
if not comfy.is_dir():
    raise SystemExit("FIX  Run cell 3 (Install) first")

print("-" * 56)
print("Getting models  ·  HOW =", how)
print("-" * 56)

hf = (cfg.get("hf_token") or secret("HF_TOKEN") or "").strip()
civit = (cfg.get("civitai_token") or secret("CIVITAI_TOKEN") or "").strip()
if hf:
    os.environ["HF_TOKEN"] = hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf

# --- optional Hugging Face download ---
CATALOG = {
    "sd15": dict(folder="checkpoints", filename="v1-5-pruned-emaonly.safetensors", size="4.3 GB",
                 url="https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"),
    "dreamshaper": dict(folder="checkpoints", filename="DreamShaper_8_pruned.safetensors", size="2.0 GB",
                        url="https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors"),
    "sdxl": dict(folder="checkpoints", filename="sd_xl_base_1.0.safetensors", size="6.9 GB",
                 url="https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"),
    "sdxl_vae": dict(folder="vae", filename="sdxl_vae.safetensors", size="0.3 GB",
                     url="https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors"),
    "lightning": dict(folder="loras", filename="sdxl_lightning_8step_lora.safetensors", size="0.4 GB",
                      url="https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_8step_lora.safetensors"),
    "canny": dict(folder="controlnet", filename="controlnet-canny-sdxl-1.0.fp16.safetensors", size="2.5 GB",
                  url="https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.fp16.safetensors"),
    "upscaler": dict(folder="upscale_models", filename="RealESRGAN_x2plus.pth", size="0.06 GB",
                     url="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"),
    "flux": dict(folder="checkpoints", filename="flux1-schnell-fp8.safetensors", size="17.2 GB",
                 url="https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors"),
}

def aria2_download(url, dest_dir: Path, filename: str):
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / filename
    if dest.is_file() and dest.stat().st_size > 1_000_000:
        print(f"  OK   already have {filename} ({dest.stat().st_size/1e9:.2f} GB)")
        return
    cmd = [
        "aria2c", "-x", "16", "-s", "16", "-k", "1M", "-c",
        "--file-allocation=none", "--console-log-level=notice", "--summary-interval=15",
        "-d", str(dest_dir), "-o", filename,
    ]
    final = url
    if "huggingface.co" in url and hf:
        cmd += [f"--header=Authorization: Bearer {hf}"]
    if "civitai.com" in url and civit:
        sep = "&" if "?" in url else "?"
        final = f"{url}{sep}token={civit}"
    cmd.append(final)
    print(f"  WAIT downloading {filename}…")
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        if "huggingface.co" in url:
            try:
                from huggingface_hub import hf_hub_download
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
                from huggingface_hub import hf_hub_download
            parts = url.split("huggingface.co/")[-1]
            repo, _, rest = parts.partition("/resolve/")
            rev_file = rest.split("/", 1)
            revision = rev_file[0] if len(rev_file) > 1 else "main"
            fname = rev_file[-1]
            hf_hub_download(repo_id=repo, filename=fname, revision=revision,
                            local_dir=str(dest_dir), token=hf or True)
            downloaded = dest_dir / fname
            if downloaded != dest and downloaded.exists():
                downloaded.replace(dest)
        else:
            subprocess.check_call(["wget", "-c", "-O", str(dest), final])
    if dest.exists():
        print(f"  OK   {filename}  ({dest.stat().st_size/1e9:.2f} GB)")
    else:
        print(f"  FIX  could not get {filename}")

if how == "download":
    flags = cfg.get("get") or {}
    queue = [(k, CATALOG[k]) for k, v in flags.items() if v and k in CATALOG]
    url = cfg.get("custom_url") or ""
    if not queue and not url:
        print("FIX  HOW is download but every GET_ line is False and CUSTOM_URL is empty.")
        print("     Go back to Your choices, set GET_DREAMSHAPER = True, Run cell 1, then this cell.")
        raise SystemExit("Nothing to download")
    print("Saving into", models_root)
    for key, item in queue:
        print(f"\n{item['filename']}  ({item['size']})")
        aria2_download(item["url"], models_root / item["folder"], item["filename"])
    if url:
        name = unquote(urlparse(url).path.rstrip("/").split("/")[-1]).split("?")[0]
        if not name or name in {"main", "resolve", "models"}:
            name = "downloaded_model.safetensors"
        print(f"\nCustom URL → {name}")
        aria2_download(url, models_root / (cfg.get("custom_folder") or "checkpoints"), name)

pulled = list(cfg.get("kagglehub_paths") or [])
if how == "kagglehub":
    slug = cfg.get("kaggle_dataset_slug") or ""
    handle = cfg.get("kaggle_model_handle") or ""
    if not slug and not handle:
        print("FIX  HOW is kagglehub but KAGGLE_DATASET_SLUG is empty.")
        print("     Paste owner/dataset-name in Your choices, Run cell 1, then this cell.")
        raise SystemExit("Paste a dataset name")
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    pulled = []
    if slug:
        print("WAIT fetching dataset", slug)
        path = kagglehub.dataset_download(slug)
        print("  OK  ", path)
        pulled.append(str(path))
    if handle:
        print("WAIT fetching model", handle)
        path = kagglehub.model_download(handle)
        print("  OK  ", path)
        pulled.append(str(path))
    save(kagglehub_paths=pulled)

# --- point ComfyUI at every source we have ---
def detect_layout(root: Path):
    models = root / "models"
    if models.is_dir() and any((models / s).is_dir() for s in SUBS):
        return "nested"
    if any((root / s).is_dir() for s in SUBS):
        return "typed"
    return "flat"

def flatten_links(src: Path, staging: Path) -> str:
    staging.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in WEIGHT_EXT:
            continue
        parent = p.parent.name.lower()
        folder = parent if parent in SUBS else "checkpoints"
        dest_dir = staging / folder
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / p.name
        if dest.exists() or dest.is_symlink():
            continue
        dest.symlink_to(p)
        n += 1
    print(f"  OK   sorted {n} loose file(s) from {src.name}")
    return str(staging)

def yaml_block(name, base: Path, prefix: str, default: bool) -> str:
    lines = [f"{name}:\n", f"    base_path: {base}/\n"]
    if default:
        lines.append("    is_default: true\n")
    for key in [
        "checkpoints", "configs", "loras", "vae", "clip_vision", "style_models",
        "embeddings", "diffusers", "vae_approx", "gligen", "upscale_models",
        "hypernetworks", "photomaker", "ipadapter", "animatediff_models",
        "animatediff_motion_lora", "model_patches", "audio_encoders",
    ]:
        lines.append(f"    {key}: {prefix}{key}/\n")
    lines.append("    text_encoders: |\n")
    lines.append(f"         {prefix}text_encoders/\n")
    lines.append(f"         {prefix}clip/\n")
    lines.append("    diffusion_models: |\n")
    lines.append(f"         {prefix}unet/\n")
    lines.append(f"         {prefix}diffusion_models/\n")
    lines.append("    controlnet: |\n")
    lines.append(f"         {prefix}controlnet/\n")
    lines.append(f"         {prefix}t2i_adapter/\n")
    return "".join(lines)

roots = []
inp = Path("/kaggle/input")
if inp.is_dir():
    kids = sorted(p for p in inp.iterdir() if p.is_dir())
    want = (cfg.get("dataset_name") or "auto").strip()
    if how == "dataset" and want and want.lower() != "auto":
        kids = [p for p in kids if p.name == want]
        if not kids:
            print("FIX  No attached dataset named", want)
            print("     Add Data on the right, or set DATASET_NAME = \"auto\"")
            raise SystemExit("Dataset not found")
    if how == "dataset" and not kids and not pulled:
        print("FIX  HOW is dataset but nothing is in /kaggle/input.")
        print("     Right sidebar → Add Data → add a dataset with .safetensors files.")
        raise SystemExit("No dataset attached")
    for src in kids:
        layout = detect_layout(src)
        n = sum(1 for p in src.rglob("*") if p.is_file() and p.suffix.lower() in WEIGHT_EXT)
        print(f"  OK   attached {src.name}  ({n} model files)")
        if layout == "flat":
            roots.append(flatten_links(src, Path("/kaggle/working/model_links") / src.name))
        else:
            roots.append(str(src))

for extra in pulled:
    p = Path(extra)
    if p.exists() and str(p) not in roots:
        layout = detect_layout(p)
        n = sum(1 for x in p.rglob("*") if x.is_file() and x.suffix.lower() in WEIGHT_EXT)
        print(f"  OK   kagglehub {p.name}  ({n} model files)")
        if layout == "flat":
            roots.append(flatten_links(p, Path("/kaggle/working/model_links") / p.name))
        else:
            roots.append(str(p))

save(dataset_roots=roots)

blocks = [yaml_block("local_store", store, "models/", default=True)]
for i, root in enumerate(roots):
    r = Path(root)
    prefix = "models/" if detect_layout(r) == "nested" else ""
    blocks.append(yaml_block(f"kaggle_data_{i}", r, prefix, default=False))
yaml_path = comfy / "extra_model_paths.yaml"
yaml_path.write_text("# Generated by ComfyUI Kaggle notebook\n" + "".join(blocks))

def replace_with_link(local: Path, target: Path):
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() or local.exists():
        if local.is_symlink() and os.path.realpath(local) == os.path.realpath(target):
            return
        if local.is_dir() and not local.is_symlink():
            for item in local.iterdir():
                dest = target / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
            shutil.rmtree(local)
        elif local.exists() or local.is_symlink():
            local.unlink()
    local.symlink_to(target, target_is_directory=True)

replace_with_link(comfy / "output", store / "output")
replace_with_link(comfy / "input", store / "input")
replace_with_link(comfy / "user", store / "user")

ckpt = list((models_root / "checkpoints").glob("*"))
print()
print("OK   ComfyUI can see the model folders")
print("     Pictures will save in", store / "output")
if ckpt:
    print("     Checkpoints in this session:")
    for p in ckpt:
        if p.is_file():
            print(f"       {p.name}")
print("NEXT Run the last cell: Start ComfyUI")



## 5 · Start ComfyUI

Press Run and **wait**. Do not stop the cell.

After 1–3 minutes a URL appears (`https://….trycloudflare.com`):

1. Copy the whole URL
2. Open a **new** browser tab
3. Paste into the address bar → Enter
4. Do **not** click the link inside Kaggle
5. Leave this cell running

To stop: Interrupt this cell.



In [ ]:
# 5 · Start ComfyUI  —  just press Run, then wait for the URL
import json, os, shutil, socket, subprocess, sys, threading, time
from pathlib import Path

CONFIG = "/kaggle/working/comfy_kaggle_config.json"
COMFY = "/kaggle/working/ComfyUI"
PORT = 8188

def load():
    if os.path.exists(CONFIG):
        with open(CONFIG) as f:
            return json.load(f)
    return {}

cfg = load()
comfy = cfg.get("comfy_path") or COMFY
if not os.path.isdir(comfy):
    print("FIX  Run cell 3 (Install) first")
    raise SystemExit("Install first")

os.chdir(comfy)
os.environ["PYTHONUNBUFFERED"] = "1"

if cfg.get("use_multigpu"):
    try:
        import torch
        n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    except Exception:
        n = 0
    devices = ",".join(str(i) for i in range(max(n, 1)))
    os.environ["CUDA_VISIBLE_DEVICES"] = devices
    print("Multi-GPU on. CUDA_VISIBLE_DEVICES =", devices)
    print("  In the UI search  multigpu")
    print("  VAELoaderMultiGPU      →", cfg.get("vae_on", "cuda:0"))
    print("  CLIPLoaderMultiGPU     →", cfg.get("te_on", "cuda:0"), "  (text encoder)")
    print("  UNETLoaderMultiGPU     →", cfg.get("unet_on", "cuda:1"), "  (the heavy model)")
    print("  Do not use CheckpointLoaderSimple")

try:
    import torch
    cuda = torch.cuda.is_available()
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if cuda else 0
except Exception:
    cuda, vram = False, 0

if not cuda:
    print("FIX  No GPU. Session options → Accelerator → GPU T4")

if vram >= 32:
    vram_flag = "--highvram"
elif vram >= 12:
    vram_flag = None
else:
    vram_flag = "--lowvram"

help_txt = ""
try:
    help_txt = subprocess.check_output(
        [sys.executable, "main.py", "--help"],
        text=True, stderr=subprocess.STDOUT, timeout=60,
    )
except Exception:
    help_txt = ""

def supported(flag: str) -> bool:
    return True if not help_txt else flag in help_txt

if vram_flag and not supported(vram_flag):
    vram_flag = None

print("-" * 56)
print("Starting ComfyUI")
print("-" * 56)
print("WAIT  First start can take 1–3 minutes. Leave this cell running.")
yaml = Path(comfy) / "extra_model_paths.yaml"
if yaml.exists():
    print("OK   Model folders are linked")
else:
    print("     No model map yet. If the UI is empty, run cell 4 first.")

subprocess.call(["pkill", "-f", "main.py --listen"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

if not shutil.which("cloudflared"):
    deb = "/tmp/cloudflared.deb"
    subprocess.check_call([
        "wget", "-q", "-O", deb,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
    ])
    subprocess.call(["dpkg", "-i", deb])

def wait_then_tunnel(port: int):
    deadline = time.time() + 180
    while time.time() < deadline:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                break
        finally:
            sock.close()
        time.sleep(0.5)
    else:
        print("FIX  ComfyUI did not open a port. Re-run this cell.")
        return
    print("WAIT Opening a public link…")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    printed = False
    assert proc.stdout is not None
    for line in proc.stdout:
        if "trycloudflare.com" in line and "http" in line:
            url = line[line.find("http"):].strip()
            print("=" * 72)
            print("  YOUR LINK  (copy the whole line)")
            print(" ", url)
            print("  1. Copy it")
            print("  2. Open a new tab")
            print("  3. Paste into the address bar, press Enter")
            print("  4. Do not click it from Kaggle")
            print("  5. Leave this cell running")
            print("=" * 72)
            printed = True
        elif "INF" in line and not printed:
            pass

threading.Thread(target=wait_then_tunnel, args=(int(PORT),), daemon=True).start()

args = [
    sys.executable, "-u", "main.py",
    "--listen", "0.0.0.0",
    "--port", str(int(PORT)),
    "--preview-method", "auto",
]
if vram_flag:
    args.append(vram_flag)
if supported("--use-pytorch-cross-attention"):
    args.append("--use-pytorch-cross-attention")
if supported("--enable-cors-header"):
    args.extend(["--enable-cors-header", "*"])

print("WAIT Launching the app…")
code = subprocess.call(args)
print("\nComfyUI stopped. Code", code)
if code == 2:
    print("FIX  Re-run this cell. If it fails again, Interrupt and Start once more.")



## If something goes wrong

| You see | Do this |
|---|---|
| No GPU | Session options → Accelerator → GPU T4 → Save → retry cell 2 |
| Internet fail | Session options → Internet → On |
| Install cell is red | Internet On, then run Install again |
| Access denied on the URL | Copy the link → new tab → paste → Enter. Do not click it |
| Blank tab after 2 minutes | Stop Start, run it again, paste the **new** URL |
| Session died tomorrow | GPU + Internet, then 1 → 2 → 3 → 4 (HOW = dataset if files are attached) → 5 |

Pictures from this session: `ComfyUI_store/output` (Files on the right). Use **Save Version** if you want them to last.

Two GPUs: in the ComfyUI graph search **multigpu**. Put VAE + CLIP on cuda:0 and UNet on cuda:1. Do not use CheckpointLoaderSimple.

